# Binary Decision Tree From Scratch


A binary decision tree classifier built entirely from scratch, with no scikit-learn model anywhere in the pipeline. What gets implemented here:

* transforming categorical variables into binary features
* counting misclassified examples in an intermediate node
* finding the best feature to split on
* growing the tree recursively with stopping conditions
* making predictions by walking the tree
* evaluating classification error
* visualising the decision at the root

The tree operates on **binary (0 or 1) features only**. That restriction avoids two complications which would otherwise dominate the implementation: multiple children per split, and threshold selection for real-valued features. Every categorical variable is one-hot encoded first, so it costs nothing here.


In [1]:
import pandas as pd

# Load the lending club dataset


The [LendingClub](https://www.lendingclub.com/) dataset: peer-to-peer loans, where the task is to predict whether a loan turns out safe or risky.


In [2]:
loans = pd.read_csv('lending-club-data.csv.gz', low_memory=False)

We reassign the labels to have +1 for a safe loan, and -1 for a risky (bad) loan.


In [3]:
if 'bad_loans' in loans.columns:
    loans['safe_loans'] = loans['bad_loans'].apply(lambda x : +1 if x==0 else -1)
    loans = loans.drop(['bad_loans'], axis=1)

Only four categorical features are used:

1. grade of the loan
2. the length of the loan term
3. the home ownership status: own, mortgage, rent
4. number of years of employment

Since this is a binary decision tree, these are converted to a binary representation by one-hot encoding in the next section.


In [4]:
features = ['grade',              # grade of the loan
            'term',               # the term of the loan
            'home_ownership',     # home_ownership status: own, mortgage or rent
            'emp_length',         # number of years of employment
           ]
target = 'safe_loans'
loans = loans[features + [target]]

Let's explore what the dataset looks like.


In [5]:
loans.columns

Index(['grade', 'term', 'home_ownership', 'emp_length', 'safe_loans'], dtype='object')

In [6]:
loans

,grade,term,home_ownership,emp_length,safe_loans
0,B,36 months,RENT,10+ years,1
1,C,60 months,RENT,< 1 year,-1
2,C,36 months,RENT,10+ years,1
3,C,36 months,RENT,10+ years,1
4,A,36 months,RENT,3 years,1
...,...,...,...,...,...
122602,E,60 months,MORTGAGE,NaN,-1
122603,D,36 months,MORTGAGE,10+ years,1
122604,D,60 months,MORTGAGE,5 years,-1
122605,D,60 months,MORTGAGE,10+ years,-1


## Subsample dataset to make sure classes are balanced


We will undersample the larger class (safe loans) in order to balance out our dataset. This means we are throwing away many data points. We use `seed=1` so everyone gets the same results.


In [7]:
safe_loans_raw = loans[loans[target] == 1]
risky_loans_raw = loans[loans[target] == -1]

# Since there are less risky loans than safe loans, find the ratio of the sizes
# and use that percentage to undersample the safe loans.
percentage = len(risky_loans_raw)/float(len(safe_loans_raw))
safe_loans = safe_loans_raw.sample(frac=percentage, random_state=1)
risky_loans = risky_loans_raw
loans_data = pd.concat([risky_loans, safe_loans], axis=0)

print("Percentage of safe loans                 :", len(safe_loans) / float(len(loans_data)))
print("Percentage of risky loans                :", len(risky_loans) / float(len(loans_data)))
print("Total number of loans in our new dataset :", len(loans_data))

Percentage of safe loans                 : 0.5
Percentage of risky loans                : 0.5
Total number of loans in our new dataset : 46300


**Note:** There are many approaches for dealing with imbalanced data, including some where we modify the learning algorithm. Some of them are reviewed in "Learning from Imbalanced Data" by Haibo He and Edwardo A. Garcia, *IEEE Transactions on Knowledge and Data Engineering* **21**(9) (June 26, 2009), p. 1263–1284. The approach used here is the simplest one: subsampling the overly represented class to get a more balanced dataset. 


## Transform categorical data into binary features


The tree handles **binary features** (a special case of categorical variables taking two values, true/false), so every categorical column needs turning into binary columns.

For instance, **home_ownership** is either `own`, `mortgage` or `rent`. A data point with

```
   {'home_ownership': 'RENT'}
```

becomes three features:

```
 {
   'home_ownership = OWN'      : 0,
   'home_ownership = MORTGAGE' : 0,
   'home_ownership = RENT'     : 1
 }
```


In [8]:
loans_data = pd.concat([risky_loans, safe_loans], axis=0)

Using pandas dataframe, to one-hot-encode, here we are using [get_dummies()](https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.get_dummies.html).


In [9]:
#one-hot-encoding

# loans_data.loc[:, features] 
loans_data = pd.get_dummies(loans_data, columns=features, prefix_sep='.', dummy_na=False).fillna(0)
loans_data

,safe_loans,grade.A,grade.B,grade.C,grade.D,grade.E,grade.F,grade.G,term. 36 months,term. 60 months,...,emp_length.10+ years,emp_length.2 years,emp_length.3 years,emp_length.4 years,emp_length.5 years,emp_length.6 years,emp_length.7 years,emp_length.8 years,emp_length.9 years,emp_length.< 1 year
1,-1,False,False,True,False,False,False,False,False,True,...,False,False,False,False,False,False,False,False,False,True
6,-1,False,False,False,False,False,True,False,False,True,...,False,False,False,True,False,False,False,False,False,False
7,-1,False,True,False,False,False,False,False,False,True,...,False,False,False,False,False,False,False,False,False,True
10,-1,False,False,True,False,False,False,False,True,False,...,False,False,False,False,False,False,False,False,False,True
12,-1,False,True,False,False,False,False,False,True,False,...,False,False,True,False,False,False,False,False,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
10016,1,True,False,False,False,False,False,False,True,False,...,False,False,True,False,False,False,False,False,False,False
58367,1,False,True,False,False,False,False,False,True,False,...,False,False,False,False,False,False,True,False,False,False
90431,1,False,True,False,False,False,False,False,True,False,...,False,False,False,False,False,False,False,True,False,False
115727,1,False,False,False,False,False,True,False,False,True,...,False,False,True,False,False,False,False,False,False,False


As can be seen above, the feature columns are now one-hot-encoded


Let's see what the feature columns look like now:


In [10]:
features = loans_data.columns
features = features.drop('safe_loans')  # Remove the response variable
features

Index(['grade.A', 'grade.B', 'grade.C', 'grade.D', 'grade.E', 'grade.F',
       'grade.G', 'term. 36 months', 'term. 60 months',
       'home_ownership.MORTGAGE', 'home_ownership.OTHER', 'home_ownership.OWN',
       'home_ownership.RENT', 'emp_length.1 year', 'emp_length.10+ years',
       'emp_length.2 years', 'emp_length.3 years', 'emp_length.4 years',
       'emp_length.5 years', 'emp_length.6 years', 'emp_length.7 years',
       'emp_length.8 years', 'emp_length.9 years', 'emp_length.< 1 year'],
      dtype='object')

In [11]:
print("Number of features (after binarizing categorical variables) = ", len(features))

Number of features (after binarizing categorical variables) =  24


Let's explore what one of these columns looks like:


In [12]:
loans_data['grade.A']

1         False
6         False
7         False
10        False
12        False
          ...  
10016      True
58367     False
90431     False
115727    False
105752    False
Name: grade.A, Length: 46300, dtype: bool

This column is set to 1 if the loan grade is A and 0 otherwise.

**Checkpoint:** Make sure the following answers match up.


In [13]:
print("Total number of grade.A loans : %s" % loans_data['grade.A'].sum())
print("Expected answer               : 6508")

Total number of grade.A loans : 6508
Expected answer               : 6508


## Train-test split

We split the data into a train test split with 80% of the data in the training set and 20% of the data in the test set. We use `seed=1` so that everyone gets the same result.


Splitting from the help of this [stackoverflow answer](https://stackoverflow.com/a/24147363/7771202)


In [14]:
import numpy as np

np.random.seed(1)
msk = np.random.rand(len(loans_data)) < 0.8 # a mask with 80% True

train_data, test_data = loans_data[msk], loans_data[~msk]

print(train_data.shape, test_data.shape)

(37038, 25) (9262, 25)


# Decision tree implementation


The tree is built from scratch in several pieces, each tested before the next is built on top of it.

## Counting mistakes when predicting the majority class

Prediction at an intermediate node works by predicting the **majority class** among all data points reaching that node.

This function counts the **misclassified examples** under that rule, which is what makes candidate splits comparable: the best split is the one leaving the fewest mistakes behind. Only the labels are needed, not the features.


In [15]:
def intermediate_node_num_mistakes(labels_in_node):
    # Corner case: If labels_in_node is empty, return 0
    if len(labels_in_node) == 0:
        return 0
    
    # Count the number of 1's (safe loans)
    num_safe = (labels_in_node == 1).sum()
    
    # Count the number of -1's (risky loans)
    num_risky = (labels_in_node == -1).sum()
                
    # Return the number of mistakes that the majority classifier makes.
    # Majority prediction will be the class with more examples.
    # All other examples at this node are mistakes.
    num_mistakes = min(num_safe, num_risky)

    return num_mistakes
    

A test case for `intermediate_node_num_mistakes`, run before anything is built on top of it.


In [16]:
# Test case 1
example_labels = pd.Series([-1, -1, 1, 1, 1])
if intermediate_node_num_mistakes(example_labels) == 2:
    print('Test passed!')
else:
    print('Test 1 failed... try again!')

# Test case 2
example_labels = pd.Series([-1, -1, 1, 1, 1, 1, 1])
if intermediate_node_num_mistakes(example_labels) == 2:
    print('Test passed!')
else:
    print('Test 2 failed... try again!')
    
# Test case 3
example_labels = pd.Series([-1, -1, -1, -1, -1, 1, 1])
if intermediate_node_num_mistakes(example_labels) == 2:
    print('Test passed!')
else:
    print('Test 3 failed... try again!')

Test passed!
Test passed!
Test passed!


## Function to pick best feature to split on


The function **best_splitting_feature** takes 3 arguments: 
1. The data (a dataframe including all of the feature columns and the label column)
2. The features to consider for splits (a list of strings of column names to consider for splits)
3. The name of the target/label column (string)

The function will loop through the list of possible features, and consider splitting on each of them. It will calculate the classification error of each split and return the feature that had the smallest classification error when split on.

The **classification error** is defined as follows:
$$
\mbox{classification error} = \frac{\mbox{# mistakes}}{\mbox{# total examples}}
$$

The steps:
* **Step 1:** Loop over each feature in the feature list
* **Step 2:** Within the loop, split the data into two groups: one group where all of the data has feature value 0 or False (we will call this the **left** split), and one group where all of the data has feature value 1 or True (we will call this the **right** split). Make sure the **left** split corresponds with 0 and the **right** split corresponds with 1 so the convention matches the tree-building code.
* **Step 3:** Calculate the number of misclassified examples in both groups of data and use the above formula to compute the **classification error**.
* **Step 4:** If the computed error is smaller than the best error found so far, store this **feature and its error**.

**Note:** Remember that since we are only dealing with binary features, we do not have to consider thresholds for real-valued features. This makes the implementation of this function much easier.


In [17]:
def best_splitting_feature(data, features, target):
    
    best_feature = None # Keep track of the best feature 
    best_error = 10     # Keep track of the best error so far 
    # Note: Since error is always <= 1, we should intialize it with something larger than 1.

    # Convert to float to make sure error gets computed correctly.
    num_data_points = float(len(data))  
    
    # Loop through each feature to consider splitting on that feature
    for feature in features:
        
        # The left split will have all data points where the feature value is 0
        left_split = data[data[feature] == 0]
        
        # The right split will have all data points where the feature value is 1
        right_split = data[data[feature] == 1]
            
        # Calculate the number of misclassified examples in the left split.
        # Remember that we implemented a function for this! (It was called intermediate_node_num_mistakes)
        left_mistakes = intermediate_node_num_mistakes(left_split[target])          

        # Calculate the number of misclassified examples in the right split.
        right_mistakes = intermediate_node_num_mistakes(right_split[target])
            
        # Compute the classification error of this split.
        # Error = (# of mistakes (left) + # of mistakes (right)) / (# of data points)
        error = float(left_mistakes + right_mistakes) / num_data_points

        # If this is the best error we have found so far, store the feature as best_feature and the error as best_error
        if error < best_error:
            best_error = error
            best_feature = feature
        
    
    return best_feature # Return the best feature we found

Testing `best_splitting_feature`:


In [18]:
if best_splitting_feature(train_data, features, 'safe_loans') == 'term. 36 months':
    print('Test passed!')
else:
    print('Test failed... try again!')

Test passed!


## Building the tree

With those functions in place, the tree itself can be built. Each node in the decision tree is represented as a dictionary which contains the following keys and possible values:

    { 
       'is_leaf'            : True/False.
       'prediction'         : Prediction at the leaf node.
       'left'               : (dictionary corresponding to the left tree).
       'right'              : (dictionary corresponding to the right tree).
       'splitting_feature'  : The feature that this node splits on.
    }

First, a function creating a leaf node from a set of target values.


In [19]:
def create_leaf(target_values):
    
    # Create a leaf node
    leaf = {'splitting_feature' : None,
            'left' : None,
            'right' : None,
            'is_leaf': True,
            'prediction': None
           }
    
    # Count the number of data points that are +1 and -1 in this node.
    num_ones = len(target_values[target_values == +1])
    num_minus_ones = len(target_values[target_values == -1])
    
    # For the leaf node, set the prediction to be the majority class.
    # Store the predicted class (1 or -1) in leaf['prediction']
    if num_ones > num_minus_ones:
        leaf['prediction'] = +1
    else:
        leaf['prediction'] = -1
        
    # Return the leaf node        
    return leaf 

The recursive learning function implements three stopping conditions:

1. **Stopping condition 1:** all data points in a node are from the same class.
2. **Stopping condition 2:** no more features to split on.
3. **Additional stopping condition:** a cap on **max_depth**. Limiting depth keeps learning cheap and stops the tree growing until every leaf is pure.


In [20]:
def decision_tree_create(data, features, target, current_depth = 0, max_depth = 10):
    remaining_features = list(features) # Make a copy of the features.
    
    target_values = data[target]
    print("--------------------------------------------------------------------")
    print("Subtree, depth = %s (%s data points)." % (current_depth, len(target_values)))
    

    # Stopping condition 1
    # (Check if there are mistakes at current node.
    # Recall you wrote a function intermediate_node_num_mistakes to compute this.)
    if  intermediate_node_num_mistakes(target_values) == 0:
        print("Stopping condition 1 reached."     )
        # If not mistakes at current node, make current node a leaf node
        return create_leaf(target_values)
    
    # Stopping condition 2 (check if there are remaining features to consider splitting on)
    if len(remaining_features) == 0:
        print("Stopping condition 2 reached."    )
        # If there are no remaining features to consider, make current node a leaf node
        return create_leaf(target_values)    
    
    # Additional stopping condition (limit tree depth)
    if current_depth >= max_depth:
        print("Reached maximum depth. Stopping for now.")
        # If the max tree depth has been reached, make current node a leaf node
        return create_leaf(target_values)

    # Find the best splitting feature (recall the function best_splitting_feature implemented above)
    splitting_feature = best_splitting_feature(data, remaining_features, target)

    
    # Split on the best feature that we found. 
    left_split = data[data[splitting_feature] == 0]
    right_split = data[data[splitting_feature] == 1]
    remaining_features.remove(splitting_feature)
    print("Split on feature %s. (%s, %s)" %
                      (splitting_feature, len(left_split), len(right_split)))
    
    # Create a leaf node if the split is "perfect"
    if len(left_split) == len(data):
        print("Creating leaf node.")
        return create_leaf(left_split[target])
    if len(right_split) == len(data):
        print("Creating leaf node.")
        return create_leaf(right_split[target])

        
    # Repeat (recurse) on left and right subtrees
    left_tree = decision_tree_create(left_split, remaining_features, target, current_depth + 1, max_depth)        
    right_tree = right_tree = decision_tree_create(right_split, remaining_features, target, current_depth + 1, max_depth)

    return {'is_leaf'          : False, 
            'prediction'       : None,
            'splitting_feature': splitting_feature,
            'left'             : left_tree, 
            'right'            : right_tree}

Here is a recursive function to count the nodes in your tree:


In [21]:
def count_nodes(tree):
    if tree['is_leaf']:
        return 1
    return 1 + count_nodes(tree['left']) + count_nodes(tree['right'])

A test case for the tree builder, checking both the node count and the structure produced.


In [22]:
small_data_decision_tree = decision_tree_create(train_data, features, 'safe_loans', max_depth = 3)
if count_nodes(small_data_decision_tree) == 11:
    print('Test passed!')
else:
    print('Test failed... try again!')
    print('Number of nodes found                :', count_nodes(small_data_decision_tree))
    print('Number of nodes that should be there : 11' )

--------------------------------------------------------------------
Subtree, depth = 0 (37038 data points).
Split on feature term. 36 months. (9383, 27655)
--------------------------------------------------------------------
Subtree, depth = 1 (9383 data points).
Split on feature grade.A. (9255, 128)
--------------------------------------------------------------------
Subtree, depth = 2 (9255 data points).
Split on feature grade.B. (8194, 1061)
--------------------------------------------------------------------
Subtree, depth = 3 (8194 data points).
Reached maximum depth. Stopping for now.
--------------------------------------------------------------------
Subtree, depth = 3 (1061 data points).
Reached maximum depth. Stopping for now.
--------------------------------------------------------------------
Subtree, depth = 2 (128 data points).
Split on feature grade.B. (128, 0)
Creating leaf node.
--------------------------------------------------------------------
Subtree, depth = 1 (2

## Build the tree!

Now that all the tests are passing, we will train a tree model on the **train_data**. Limit the depth to 6 (**max_depth = 6**) to make sure the algorithm doesn't run for too long. Call this tree **my_decision_tree**. 

**Warning**: This code block may take 1-2 minutes to learn. 


In [23]:
# Make sure to cap the depth at 6 by using max_depth = 6
features = list(features)

my_decision_tree = decision_tree_create(
    train_data,
    features,
    target='safe_loans',
    current_depth=0,
    max_depth=6
)

--------------------------------------------------------------------
Subtree, depth = 0 (37038 data points).
Split on feature term. 36 months. (9383, 27655)
--------------------------------------------------------------------
Subtree, depth = 1 (9383 data points).
Split on feature grade.A. (9255, 128)
--------------------------------------------------------------------
Subtree, depth = 2 (9255 data points).
Split on feature grade.B. (8194, 1061)
--------------------------------------------------------------------
Subtree, depth = 3 (8194 data points).


Split on feature grade.C. (5955, 2239)
--------------------------------------------------------------------
Subtree, depth = 4 (5955 data points).
Split on feature grade.D. (3857, 2098)
--------------------------------------------------------------------
Subtree, depth = 5 (3857 data points).
Split on feature home_ownership.OTHER. (3856, 1)
--------------------------------------------------------------------
Subtree, depth = 6 (3856 data points).
Reached maximum depth. Stopping for now.
--------------------------------------------------------------------
Subtree, depth = 6 (1 data points).
Stopping condition 1 reached.
--------------------------------------------------------------------
Subtree, depth = 5 (2098 data points).


Split on feature grade.E. (2098, 0)
Creating leaf node.
--------------------------------------------------------------------
Subtree, depth = 4 (2239 data points).


Split on feature emp_length.4 years. (2106, 133)
--------------------------------------------------------------------
Subtree, depth = 5 (2106 data points).


Split on feature grade.D. (2106, 0)
Creating leaf node.
--------------------------------------------------------------------
Subtree, depth = 5 (133 data points).
Split on feature home_ownership.MORTGAGE. (65, 68)
--------------------------------------------------------------------
Subtree, depth = 6 (65 data points).
Reached maximum depth. Stopping for now.
--------------------------------------------------------------------
Subtree, depth = 6 (68 data points).
Reached maximum depth. Stopping for now.
--------------------------------------------------------------------
Subtree, depth = 3 (1061 data points).
Split on feature emp_length.3 years. (974, 87)
--------------------------------------------------------------------
Subtree, depth = 4 (974 data points).
Split on feature home_ownership.OWN. (899, 75)
--------------------------------------------------------------------
Subtree, depth = 5 (899 data points).


Split on feature emp_length.< 1 year. (816, 83)
--------------------------------------------------------------------
Subtree, depth = 6 (816 data points).
Reached maximum depth. Stopping for now.
--------------------------------------------------------------------
Subtree, depth = 6 (83 data points).
Reached maximum depth. Stopping for now.
--------------------------------------------------------------------
Subtree, depth = 5 (75 data points).
Split on feature emp_length.10+ years. (52, 23)
--------------------------------------------------------------------
Subtree, depth = 6 (52 data points).
Reached maximum depth. Stopping for now.
--------------------------------------------------------------------
Subtree, depth = 6 (23 data points).
Reached maximum depth. Stopping for now.
--------------------------------------------------------------------
Subtree, depth = 4 (87 data points).


Split on feature home_ownership.OWN. (79, 8)
--------------------------------------------------------------------
Subtree, depth = 5 (79 data points).
Split on feature grade.C. (79, 0)
Creating leaf node.
--------------------------------------------------------------------
Subtree, depth = 5 (8 data points).
Split on feature grade.C. (8, 0)
Creating leaf node.
--------------------------------------------------------------------
Subtree, depth = 2 (128 data points).
Split on feature grade.B. (128, 0)
Creating leaf node.
--------------------------------------------------------------------
Subtree, depth = 1 (27655 data points).
Split on feature grade.D. (22962, 4693)
--------------------------------------------------------------------
Subtree, depth = 2 (22962 data points).
Split on feature grade.E. (21677, 1285)
--------------------------------------------------------------------
Subtree, depth = 3 (21677 data points).
Split on feature grade.F. (21317, 360)
-----------------------------

Split on feature home_ownership.RENT. (3466, 3636)
--------------------------------------------------------------------
Subtree, depth = 6 (3466 data points).
Reached maximum depth. Stopping for now.
--------------------------------------------------------------------
Subtree, depth = 6 (3636 data points).
Reached maximum depth. Stopping for now.
--------------------------------------------------------------------
Subtree, depth = 4 (360 data points).
Split on feature emp_length.8 years. (347, 13)
--------------------------------------------------------------------
Subtree, depth = 5 (347 data points).
Split on feature grade.A. (347, 0)
Creating leaf node.
--------------------------------------------------------------------
Subtree, depth = 5 (13 data points).
Split on feature home_ownership.OWN. (8, 5)
--------------------------------------------------------------------
Subtree, depth = 6 (8 data points).
Reached maximum depth. Stopping for now.
---------------------------------------

## Making predictions with a decision tree

Predictions come from the decision tree with a simple recursive function. Below, we call this function `classify`, which takes in a learned `tree` and a test point `x` to classify.  We include an option `annotate` that describes the prediction path when set to `True`.


In [24]:
def classify(tree, x, annotate = False):   
    # if the node is a leaf node.
    if tree['is_leaf']:
        if annotate: 
            print("At leaf, predicting %s" % tree['prediction'])
        return tree['prediction'] 
    else:
        # split on feature.
        split_feature_value = x[tree['splitting_feature']]
        if annotate: 
            print("Split on %s = %s" % (tree['splitting_feature'], split_feature_value))
        if split_feature_value == 0:
            return classify(tree['left'], x, annotate)
        else:
            return classify(tree['right'], x, annotate)

Now, let's consider the first example of the test set and see what `my_decision_tree` model predicts for this data point.


In [25]:
test_data.iloc[0]

safe_loans                    -1
grade.A                    False
grade.B                    False
grade.C                    False
grade.D                     True
grade.E                    False
grade.F                    False
grade.G                    False
term. 36 months            False
term. 60 months             True
home_ownership.MORTGAGE    False
home_ownership.OTHER       False
home_ownership.OWN         False
home_ownership.RENT         True
emp_length.1 year          False
emp_length.10+ years       False
emp_length.2 years         False
emp_length.3 years         False
emp_length.4 years         False
emp_length.5 years          True
emp_length.6 years         False
emp_length.7 years         False
emp_length.8 years         False
emp_length.9 years         False
emp_length.< 1 year        False
Name: 58, dtype: object

In [26]:
print('Predicted class: %s ' % classify(my_decision_tree, test_data.iloc[0]))

Predicted class: -1 


Let's add some annotations to our prediction to see what the prediction path was that lead to this predicted class:


In [27]:
classify(my_decision_tree, test_data.iloc[0], annotate=True)

Split on term. 36 months = False
Split on grade.A = False
Split on grade.B = False
Split on grade.C = False
Split on grade.D = True
At leaf, predicting -1


-1

**The first split on this path.** The first line of the trace is `Split on term. 36 months = False`, so loan term is the feature the tree examines before anything else.


**The first split that goes right.** Walking down the path, the first split evaluating to True is `Split on grade.D = True`, so `grade.D` is the first feature sending this example right.


**The last split before the leaf.** After `Split on grade.D = True` the next line is `At leaf, predicting -1`, so `grade.D` is also the last feature split on before the prediction.


## Evaluating your decision tree


Now, we will write a function to evaluate a decision tree by computing the classification error of the tree on the given dataset.

The **classification error** is defined as follows:
$$
\mbox{classification error} = \frac{\mbox{# mistakes}}{\mbox{# total examples}}
$$

`evaluate_classification_error` takes:
1. `tree` (as described above)
2. `data` (a dataframe)
3. `target` (a string - the name of the target/label column)

It calculates a prediction (class label) for each row in `data` using the decision `tree` and return the classification error computed using the above formula.


In [28]:
def evaluate_classification_error(tree, data, target):
    # Apply the classify(tree, x) to each row in your data
    prediction = data.apply(lambda x: classify(tree,x),axis = 1)
    # Once you've made the predictions, calculate the classification error and return it
    num_mistakes = (prediction != data[target]).sum()
    return num_mistakes / float(len(data))
    

Now, let's use this function to evaluate the classification error on the test set.


In [29]:
evaluate_classification_error(my_decision_tree, test_data, target)

np.float64(0.3787518894407255)

**Classification error on the test set: 0.38** (0.3788 exactly).

The dataset was deliberately balanced 50/50 earlier, so chance sits at 0.50 and a majority-class baseline would too. An error of 0.38 means about 62% accuracy, roughly 12 points above chance, from four categorical variables and a depth-6 tree.


## Printing out a decision stump


A single decision stump can be printed to inspect the split at any node. Printing the entire tree is left as an exercise.


In [30]:
def print_stump(tree, name = 'root'):
    split_name = tree['splitting_feature'] # split_name is something like 'term. 36 months'
    if split_name is None:
        print("(leaf, label: %s)" % tree['prediction'])
        return None
    split_feature, split_value = split_name.split('.')
    print('                       %s' % name)
    print('         |---------------|----------------|')
    print('         |                                |')
    print('         |                                |')
    print('         |                                |')
    print('  [{0} == 0]               [{0} == 1]    '.format(split_name))
    print('         |                                |')
    print('         |                                |')
    print('         |                                |')
    print('    (%s)                         (%s)' % (('leaf, label: ' +
          str(tree['left']['prediction']) if tree['left']['is_leaf'] else 'subtree'),
           ('leaf, label: ' + str(tree['right']['prediction']) if tree['right']['is_leaf'] else 'subtree'))
         )

In [31]:
print_stump(my_decision_tree)

                       root
         |---------------|----------------|
         |                                |
         |                                |
         |                                |
  [term. 36 months == 0]               [term. 36 months == 1]    
         |                                |
         |                                |
         |                                |
    (subtree)                         (subtree)


**The root split is `term. 36 months`.** Of the 24 binary features, loan term is the single most informative, ahead of every loan grade.

### Exploring the intermediate left subtree

The tree is a recursive dictionary, so every node is reachable:
* `my_decision_tree['left']` to go left
* `my_decision_tree['right']` to go right


In [32]:
print_stump(my_decision_tree['left'], my_decision_tree['splitting_feature'])

                       term. 36 months
         |---------------|----------------|
         |                                |
         |                                |
         |                                |
  [grade.A == 0]               [grade.A == 1]    
         |                                |
         |                                |
         |                                |
    (subtree)                         (leaf, label: 1)


### Exploring the left subtree of the left subtree


In [33]:
print_stump(my_decision_tree['left']['left'], my_decision_tree['left']['splitting_feature'])

                       grade.A
         |---------------|----------------|
         |                                |
         |                                |
         |                                |
  [grade.B == 0]               [grade.B == 1]    
         |                                |
         |                                |
         |                                |
    (subtree)                         (subtree)


**First three splits down the left-most branch.** Starting at the root and always going left:

- `term. 36 months == 0`
- `grade.A == 0`
- `grade.B == 0`

The tree checks loan grades in alphabetical order, which is also their quality order. Grade arrived as one-hot dummies carrying no ordering information, so the tree is recovering that ordinal scale on its own, purely from how the labels separate.


In [34]:
print_stump(my_decision_tree['right'], my_decision_tree['splitting_feature'])
print_stump(my_decision_tree['right']['right'], my_decision_tree['right']['splitting_feature'])

                       term. 36 months
         |---------------|----------------|
         |                                |
         |                                |
         |                                |
  [grade.D == 0]               [grade.D == 1]    
         |                                |
         |                                |
         |                                |
    (subtree)                         (leaf, label: -1)
(leaf, label: -1)


**First three splits down the right-most branch.** Starting at the root and always going right:

- `term. 36 months == 1`
- `grade.D == 1`
- leaf

The right side terminates far sooner. A 36-month loan of grade D is apparently decidable without further questions.


**Credit: Emily Fox & Carlos Guestrin, Washington University**
